# Data Preprocessing
This notebook cleans and normalizes all 4 datasets and combines them into a single labeled CSV for model training.

## Output
`data/labeled/dataset.csv` with columns:
- `text` — cleaned document text
- `label` — category (news, email, contract, invoice)

In [1]:
import os
import re
import pandas as pd
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))
MAX_WORDS = 500

os.chdir('..')

# Base cleaning function - shared by all datasets
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# News specific cleaning
def clean_news(text):
    text = clean_text(text)
    text = re.sub(r'\b\d+\w*\b', '', text)    # remove numbers AND number+letter combos like 600m, 113bn
    text = re.sub(r'\b\w\b', '', text)          # remove single characters
    words = text.split()
    words = [w for w in words if w not in STOPWORDS]
    words = words[:MAX_WORDS]
    return ' '.join(words).strip()

# Email specific cleaning
def clean_email(text):
    # Remove email server headers
    text = re.sub(r'received.*?esmtp.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'mime version.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'content type.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'content transfer.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'message id.*?\n', '', text, flags=re.IGNORECASE)
    # Remove forwarded message chains
    text = re.sub(r'-+\s*forwarded by.*?am|pm', '', text, flags=re.IGNORECASE|re.DOTALL)
    # Remove "original message" blocks
    text = re.sub(r'original message.*', '', text, flags=re.IGNORECASE|re.DOTALL)
    # Remove "sent on/by" date lines
    text = re.sub(r'sent\s+(monday|tuesday|wednesday|thursday|friday|saturday|sunday).*', '', text, flags=re.IGNORECASE)
    # Remove cc and to lines
    text = re.sub(r'^(to|cc|bcc):\s.*', '', text, flags=re.IGNORECASE|re.MULTILINE)
    # Remove dashes
    text = re.sub(r'-+', ' ', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove table formatting
    text = re.sub(r'\|', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Apply base cleaning
    text = clean_text(text)
    # Remove numbers
    text = re.sub(r'\b\d+\w*\b', '', text)
    # Remove single characters
    text = re.sub(r'\b\w\b', '', text)
    words = text.split()
    words = [w for w in words if w not in STOPWORDS]
    words = words[:MAX_WORDS]
    return ' '.join(words).strip()

print("✅ Ready to preprocess!")
print(f"📁 Working directory: {os.getcwd()}")

✅ Ready to preprocess!
📁 Working directory: c:\Users\Adri\Documents\IE University\3rd Year\Second Semester\Statistical Learning\Nueva carpeta\Document-Classification-and-Information-Extraction-


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 1. Preprocess BBC News

In [2]:
news_data = []
bbc_path = "data/raw/news/bbc"

for category in os.listdir(bbc_path):
    category_path = f"{bbc_path}/{category}"
    if os.path.isdir(category_path):
        for filename in os.listdir(category_path):
            filepath = f"{category_path}/{filename}"
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.read().split('\n')
                title = lines[0].strip()
                body = ' '.join(lines[1:]).strip()
                full_text = title + ' ' + body
                cleaned = clean_news(full_text)
                news_data.append({'text': cleaned, 'label': 'news'})

df_news = pd.DataFrame(news_data)
df_news = df_news.drop_duplicates(subset='text')
df_news = df_news[df_news['text'].apply(lambda x: len(x.split()) >= 20)] 

print(f"📰 News articles loaded: {len(df_news)}")
print(f"\n📰 Sample cleaned article:\n{df_news['text'][0]}")

📰 News articles loaded: 2119

📰 Sample cleaned article:
ad sales boost time warner profit quarterly profits us media giant timewarner jumped three months december yearearlier firm one biggest investors google benefited sales highspeed internet connections higher advert sales timewarner said fourth quarter sales rose profits buoyed oneoff gains offset profit dip warner bros less users aol time warner said friday owns searchengine google internet business aol mixed fortunes lost subscribers fourth quarter profits lower preceding three quarters however company said aols underlying profit exceptional items rose back stronger internet advertising revenues hopes increase subscribers offering online service free timewarner internet customers try sign aols existing customers highspeed broadband timewarner also restate results following probe us securities exchange commission sec close concluding time warners fourth quarter profits slightly better analysts expectations film division saw profits

## 2. Preprocess Enron Emails

In [ ]:
# Filter function for system generated emails
SYSTEM_KEYWORDS = ['sql error', 'parsing file', 'locked user', 'admin machine', 
                   'log messages', 'error update', 'schedule found', 'final schedule', 
                   'hourahead hour', 'esmtp', 'mime version', 'rly ygol',
                   'content type text', 'smtp id']

def is_system_email(text):
    return any(keyword in text.lower() for keyword in SYSTEM_KEYWORDS)

# Load raw emails
df_raw_emails = pd.read_csv("data/raw/emails/enron_spam_data.csv")

# Combine subject and message
df_raw_emails['combined'] = df_raw_emails['Subject'].fillna('') + ' ' + df_raw_emails['Message'].fillna('')

# Clean using email specific function
df_raw_emails['cleaned'] = df_raw_emails['combined'].apply(clean_email)

# Add word count
df_raw_emails['word_count'] = df_raw_emails['cleaned'].apply(lambda x: len(x.split()))

# Filter system emails and short emails
df_raw_emails['is_system'] = df_raw_emails['cleaned'].apply(is_system_email)
df_filtered = df_raw_emails[(df_raw_emails['word_count'] >= 50) & (~df_raw_emails['is_system'])]

print(f"📧 After filtering: {len(df_filtered)} usable emails")

# Sample 2000
df_emails = df_filtered[['cleaned']].rename(columns={'cleaned': 'text'}).sample(2000, random_state=42)
df_emails['label'] = 'email'
df_emails = df_emails.drop_duplicates(subset='text')
df_emails = df_emails.reset_index(drop=True)

print(f"📧 Emails loaded: {len(df_emails)}")
print(f"\n📧 Sample cleaned email:\n{df_emails['text'].iloc[0]}")

📧 After filtering: 13320 usable emails
📧 Emails loaded: 1228

📧 Sample cleaned email:
added role thanks sally great brent appreciate stretching north american operation facilitate look forward speaking soon john enron capital trade resources corp sally beck john sherriff lon ect ect cc subject added role congratulations added role ceo enron europe look forward continuing work staff london risk management operations initiatives talk brent price several times week know fernley appreciated participation london hope found involvement beneficial well almost half way mark assignment talk next week return calgary get input role date


In [6]:
for i in range(5):
    print(f"--- Email {i+1} ---")
    print(df_emails['text'].iloc[i])
    print()

--- Email 1 ---
added role thanks sally great brent appreciate stretching north american operation facilitate look forward speaking soon john enron capital trade resources corp sally beck john sherriff lon ect ect cc subject added role congratulations added role ceo enron europe look forward continuing work staff london risk management operations initiatives talk brent price several times week know fernley appreciated participation london hope found involvement beneficial well almost half way mark assignment talk next week return calgary get input role date

--- Email 2 ---
fwd mark market return path received rly ygol mx aol com rly ygol mail aol com air ygo mail aol com bl esmtp fri jan received mailman enron com mailman enron com rly ygol mx aol com bl esmtp fri jan received dservl ect enron com dservl ect enron com mailman enron com corp esmtp id xaal fri jan gmt received notes ect enron com notes ect enron com dservl ect enron com smtp id raa fri jan cst received notes ect enron c